In [ ]:
# this notebook will prepare/ clean data for ML model
import json
import pandas as pd
import gc
from pathlib import Path
import numpy as np

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "bigData"

PATH_15m = Path.cwd().parents[1] / "data" / "bigData" / "BTCUSDT-15m.json"
PATH_5m = Path.cwd().parents[1] / "data" / "bigData" / "BTCUSDT-5m.json"

with open(PATH_15m) as f:
    file = json.load(f)

ohlcv = file["ohlcv"]  # list of [timestamp, open, high, low, close, volume]
del file  # free the rest of the JSON (symbol, onchart, etc.), , preserve RAM

df_15 = pd.DataFrame(ohlcv, columns=["timestamp", "open", "high", "low", "close", "vol"])
del ohlcv  # free the Python list-of-lists, preserve RAM
gc.collect()

with open(PATH_5m) as f:
    file = json.load(f)

ohlcv = file["ohlcv"]  # list of [timestamp, open, high, low, close, volume]
del file  # free the rest of the JSON (symbol, onchart, etc.), , preserve RAM

df_5 = pd.DataFrame(ohlcv, columns=["timestamp", "open", "high", "low", "close", "vol"])
del ohlcv  # free the Python list-of-lists, preserve RAM
gc.collect()

# Halve memory: float64 -> float32 for price/volume columns
df_15[["open", "high", "low", "close", "vol"]] = df_15[["open", "high", "low", "close", "vol"]].astype("float32")
df_5[["open", "high", "low", "close", "vol"]] = df_5[["open", "high", "low", "close", "vol"]].astype("float32")

# code was truncated by the NotebookEdit tool on the previous write
print(f"15m Shape: {df_15.shape} | Memory: {df_15.memory_usage(deep=True).sum()/ 1e6:.2f} MB") # 2.15 MB
print(f"5m Shape: {df_5.shape} | Memory: {df_5.memory_usage(deep=True).sum()/ 1e6:.2f} MB") # 6.41 MB

df_5 = df_5.sort_values('timestamp').reset_index(drop=True)
df_15 = df_15.sort_values('timestamp').reset_index(drop=True)

15m Shape: (76924, 6) | Memory: 2.15 MB
5m Shape: (228772, 6) | Memory: 6.41 MB


# Version 6 use 5m, 15m and 45m(aggregated)

# 5m data - ATR42
use ATR 42

In [2]:
# ATR42 on 5m — equivalent to ATR14 on 15m in time

prev_close_5m = df_5['close'].shift(1)
tr_5m = pd.concat([
    df_5['high'] - df_5['low'],
    (df_5['high'] - prev_close_5m).abs(),
    (df_5['low']  - prev_close_5m).abs(),
], axis=1).max(axis=1)

df_5['atr42'] = tr_5m.rolling(42).mean().astype('float32')

print(f"ATR42 range: {df_5['atr42'].min():.2f} – {df_5['atr42'].max():.2f}")
df_5[['timestamp', 'close', 'atr42']].tail(3)


ATR42 range: 7.42 – 1552.81


,timestamp,close,atr42
228769,1772668200000,72668.960938,162.623138
228770,1772668500000,72666.773438,161.973770
228771,1772668800000,72677.328125,162.421127


# 5m - Triple Barrier
use ltf_arr from now on

In [3]:
# triple-barrier on 5m (gap-aware)
FIVE_MIN_MS = 300_000

# Snap 5m timestamps to grid (absorbs ms jitter) — needed for gap detection
df_5['ts_5m'] = (df_5['timestamp'] // FIVE_MIN_MS) * FIVE_MIN_MS

# Triple-barrier on 5m (gap-aware)
ATR_MULT    = 1.0
MAX_BARS_5M = 3   # 1h lookahead — tune: 6=30m, 12=1h, 24=2h, 48=4h

# df_5   = ltf_df.reset_index(drop=True)

high_arr  = df_5['high'].values
low_arr   = df_5['low'].values
open_arr  = df_5['open'].values
close_arr = df_5['close'].values
atr_arr   = df_5['atr42'].values
ts5_arr   = df_5['ts_5m'].values

labels = np.full(len(df_5), np.nan)

for i in range(len(df_5)):
    if np.isnan(atr_arr[i]):
        continue

    entry = close_arr[i]
    tp    = atr_arr[i] * ATR_MULT
    sl    = atr_arr[i] * ATR_MULT
    label = 0  # default: timeout

    for k in range(1, MAX_BARS_5M + 1):
        j = i + k
        if j >= len(df_5):
            break
        # Stop at data gap — don't evaluate across market closures
        if ts5_arr[j] != ts5_arr[i] + k * FIVE_MIN_MS:
            break

        h, l, o = high_arr[j], low_arr[j], open_arr[j]
        hit_up   = (h - entry) >= tp
        hit_down = (entry - l) >= sl

        if hit_up and hit_down:
            # Both barriers in same bar: open proximity infers direction first
            label = 1 if abs(o - (entry + tp)) < abs(o - (entry - sl)) else -1
            break
        elif hit_up:
            label = 1
            break
        elif hit_down:
            label = -1
            break

    labels[i] = label

df_5['label'] = labels

# Stats
total  = df_5['label'].notna().sum()
n_up   = (df_5['label'] ==  1).sum()
n_down = (df_5['label'] == -1).sum()
n_time = (df_5['label'] ==  0).sum()
n_nan  = df_5['label'].isna().sum()

print(f"Total labeled  : {total:,}")
print(f"  Up   ( 1)    : {n_up:,}  ({n_up   / total * 100:.1f}%)")
print(f"  Down (-1)    : {n_down:,}  ({n_down / total * 100:.1f}%)")
print(f"  Timeout (0)  : {n_time:,}  ({n_time  / total * 100:.1f}%)  ← masked during training")
print(f"  NaN (warmup) : {n_nan:,}")


Total labeled  : 228,731
  Up   ( 1)    : 76,220  (33.3%)
  Down (-1)    : 77,821  (34.0%)
  Timeout (0)  : 74,690  (32.7%)  ← masked during training
  NaN (warmup) : 41


In [4]:
# keep all rows, add train_mask
# Keep full sequence intact — LSTM needs contiguous rows
# train_mask = 1: valid label, backprop on this step
# train_mask = 0: timeout or warmup NaN, zero out loss contribution

df_5['train_mask'] = ((df_5['label'] == 1) | (df_5['label'] == -1)).astype('int8')

# Fill NaN warmup labels with 0 so the array stays numeric
df_5['label'] = df_5['label'].fillna(0).astype('int8')

trainable = df_5['train_mask'].sum()
total     = len(df_5)

print(f"Total rows     : {total:,}")
print(f"  Trainable    : {trainable:,}  ({trainable / total * 100:.1f}%)")
print(f"  Masked out   : {total - trainable:,}  (timeout + warmup NaN)")
masked_in = df_5[df_5['train_mask'] == 1]
print(f"Label balance (trainable): {masked_in['label'].value_counts().to_dict()}")
df_5.tail()



Total rows     : 228,772
  Trainable    : 154,041  (67.3%)
  Masked out   : 74,731  (timeout + warmup NaN)
Label balance (trainable): {-1: 77821, 1: 76220}


,timestamp,open,high,low,close,vol,atr42,ts_5m,label,train_mask
228767,1772667600000,72701.101562,72756.812500,72655.328125,72756.796875,56.436958,166.503159,1772667600000,0,0
228768,1772667900000,72756.812500,72830.007812,72701.179688,72701.187500,43.030441,164.771942,1772667900000,0,0
228769,1772668200000,72701.179688,72719.687500,72603.843750,72668.960938,169.471603,162.623138,1772668200000,1,1
228770,1772668500000,72668.953125,72775.468750,72639.679688,72666.773438,275.874146,161.973770,1772668500000,1,1
228771,1772668800000,72666.773438,72854.000000,72657.953125,72677.328125,168.545258,162.421127,1772668800000,0,0


# 5m - identities
group A - overlays

In [5]:
# --- Group A: 5m OHLCV features (on ltf_arr, so they live alongside label + train_mask) ---
df_5['body_pct']   = (df_5['close'] - df_5['open']) / df_5['open']   # candle direction + size
df_5['vol_norm']   = df_5['vol'] / df_5['vol'].rolling(96).mean()        # relative vol (vs 8h avg)
df_5['close_ret1'] = df_5['close'].pct_change(1)                            # 1-bar return
df_5['atr42_pct']  = df_5['atr42'] / df_5['close']                      # volatility regime (normalised, non-stationary raw excluded)

# Add 15m and 45m features
**ablation V6-F**

In [6]:
def _fix_break_of_structure(isXXX, isSource, high, low):
    """
    Ensure isXXX alternates strictly between 1 and -1.
    On a break (1,1 or -1,-1), insert the best counter swing between the two offending positions,
    but ONLY if it is structurally valid:
      - Inserting a  1 (high): candidate high must be > max(low[p1],  low[p2])
      - Inserting a -1 (low) : candidate low  must be < min(high[p1], high[p2])
    If no valid candidate exists, remove the less extreme of the two duplicates instead.
    """
    changed = True
    while changed:
        changed = False
        non_zero = np.where(isXXX != 0)[0]

        for i in range(1, len(non_zero)):
            p1, p2 = non_zero[i - 1], non_zero[i]
            v1, v2 = isXXX[p1], isXXX[p2]

            if v1 != v2:
                continue

            counter    = np.int8(-v1)
            candidates = np.arange(p1 + 1, p2)

            inserted = False
            if len(candidates) > 0:
                source_cands = candidates[isSource[candidates] == counter]
                pool = source_cands if len(source_cands) > 0 else candidates

                if counter == -1:
                    # Inserting a low: must dip below both surrounding highs
                    threshold  = min(high[p1], high[p2])
                    valid_pool = pool[low[pool] < threshold]
                else:
                    # Inserting a high: must rise above both surrounding lows
                    threshold  = max(low[p1], low[p2])
                    valid_pool = pool[high[pool] > threshold]

                if len(valid_pool) > 0:
                    best = (valid_pool[np.argmin(low[valid_pool])]
                            if counter == -1
                            else valid_pool[np.argmax(high[valid_pool])])
                    isXXX[best] = counter
                    inserted = True

            if not inserted:
                # No structurally valid candidate — drop the less extreme duplicate
                if v1 == 1:   # two highs: keep the higher one
                    isXXX[p1 if high[p1] < high[p2] else p2] = 0
                else:          # two lows: keep the lower one
                    isXXX[p1 if low[p1]  > low[p2] else p2] = 0

            changed = True
            break  # restart after each change

    return isXXX


In [7]:
def _fix_price_order(isXXX, high, low):
    """
    Fix price ordering violations by inserting 2 bridging marks instead of removing.
    For ITR/LTR: inserted marks need not be original swing points.
    Falls back to removal only if no valid bridge candidates exist.
    """
    changed = True
    while changed:
        changed = False
        non_zero = np.where(isXXX != 0)[0]

        for i in range(1, len(non_zero)):
            p1, p2 = non_zero[i-1], non_zero[i]
            v1, v2 = isXXX[p1], isXXX[p2]

            if v1 == 1 and v2 == -1 and low[p2] >= high[p1]:
                # Need a real low then a real high between p1 and p2
                cands = np.arange(p1 + 1, p2)
                valid_lows = cands[low[cands] < high[p1]]
                if len(valid_lows) > 0:
                    best_low = valid_lows[np.argmin(low[valid_lows])]
                    after = np.arange(best_low + 1, p2)
                    valid_highs = after[high[after] > low[best_low]]
                    if len(valid_highs) > 0:
                        best_high = valid_highs[np.argmax(high[valid_highs])]
                        isXXX[best_low]  = -1
                        isXXX[best_high] = 1
                        changed = True
                        break
                # fallback: remove the bad low
                isXXX[p2] = 0
                changed = True
                break

            elif v1 == -1 and v2 == 1 and high[p2] <= low[p1]:
                # Need a real high then a real low between p1 and p2
                cands = np.arange(p1 + 1, p2)
                valid_highs = cands[high[cands] > low[p1]]
                if len(valid_highs) > 0:
                    best_high = valid_highs[np.argmax(high[valid_highs])]
                    after = np.arange(best_high + 1, p2)
                    valid_lows = after[low[after] < high[best_high]]
                    if len(valid_lows) > 0:
                        best_low = valid_lows[np.argmin(low[valid_lows])]
                        isXXX[best_high] = 1
                        isXXX[best_low]  = -1
                        changed = True
                        break
                # fallback: remove the bad high
                isXXX[p2] = 0
                changed = True
                break

    return isXXX


In [8]:
def strHunt(df, window=4):
    n = len(df)
    df = df.copy()

    high = df['high'].values
    low  = df['low'].values

    isSTR = np.zeros(n, dtype=np.int8)

    # --- First window: can't look backward, so determine direction from the span ---
    # if n > window + 1:
    #     diff = high[window + 1] - low[0]
    #     if diff > 0:
    #         isSTR[0] = -1   # start of bullish move → mark as low
    #     elif diff < 0:
    #         isSTR[0] = 1    # start of bearish move → mark as high

    # --- First window: method 2 just mark 0

    # --- Main loop: candles with a full symmetric window on both sides ---
    # Stop at n-window so the last `window` candles remain 0
    for i in range(window, n - window):
        win_high = high[i - window : i + window + 1]
        win_low  = low [i - window : i + window + 1]

        is_swing_high = high[i] >= win_high.max()   # local maximum
        is_swing_low  = low[i]  <= win_low.min()    # local minimum

        if is_swing_high and not is_swing_low:
            isSTR[i] = 1    # confirmed swing high
        elif is_swing_low and not is_swing_high:
            isSTR[i] = -1   # confirmed swing low
        # neither → stays 0
        # both, liquidity swept on both high and low → stays 0
        

    isSTR = _fix_break_of_structure(isSTR, isSTR, high, low)
    isSTR = _fix_price_order(isSTR, high, low)

    df['isSTR'] = isSTR
    return df



In [9]:
# execute -- 
# create 45m tf from 15m
# Aggregate 15m → 45m (3 bars per bucket)
FORTY_FIVE_MIN_MS = 2_700_000

df_15 = df_15.sort_values('timestamp').reset_index(drop=True)
df_15['ts_45m'] = (df_15['timestamp'] // FORTY_FIVE_MIN_MS) * FORTY_FIVE_MIN_MS

df_45 = df_15.groupby('ts_45m', sort=True).agg(
    timestamp=('timestamp', 'first'),
    open=('open',   'first'),
    high=('high',   'max'),
    low=('low',    'min'),
    close=('close', 'last'),
    vol=('vol',    'sum'),
).reset_index(drop=True)

df_45[['open','high','low','close','vol']] = df_45[['open','high','low','close','vol']].astype('float32')

# Apply strHunt on 45m
# apply strHunt
df_15 = strHunt(df_15)
df_45 = strHunt(df_45)

marks = df_45['isSTR'].value_counts().to_dict()
print(f"45m shape: {df_45.shape}")
print(f"45m isSTR marks: {marks}  (highs: {marks.get(1,0)}, lows: {marks.get(-1,0)})")

marks = df_15['isSTR'].value_counts().to_dict()
print(f"15m shape: {df_15.shape}")
print(f"15m isSTR marks: {marks}  (highs: {marks.get(1,0)}, lows: {marks.get(-1,0)})")

45m shape: (25642, 7)
45m isSTR marks: {0: 20868, -1: 2387, 1: 2387}  (highs: 2387, lows: 2387)
15m shape: (76924, 8)
15m isSTR marks: {0: 63204, -1: 6860, 1: 6860}  (highs: 6860, lows: 6860)


In [10]:
def validate_markers(df, col):
    """
    Validate a marker column (e.g. 'isITR', 'isLTR') across the entire DataFrame.

    Checks two invariants:
      1. BOS (Break-of-Structure): consecutive non-zero entries must alternate sign (no 1,1 or -1,-1).
      2. Price order: after a high (1), the next low (-1) must be strictly below it,
         and after a low (-1), the next high (1) must be strictly above it.

    Returns
    -------
    dict with keys:
        'bos_violations'   : DataFrame of rows where sign repeats
        'order_violations' : DataFrame of rows where price order is broken
        'is_valid'         : True only if both checks pass
    """
    rows = df[df[col] != 0][['timestamp', 'high', 'low', col]].copy()
    rows['price_at_mark'] = rows.apply(
        lambda r: r['high'] if r[col] == 1 else r['low'], axis=1
    )

    vals = rows[col].values

    # --- BOS check: consecutive same-sign entries ---
    bos_mask = [False] + [vals[i] == vals[i - 1] for i in range(1, len(vals))]
    bos_violations = rows[bos_mask]

    # --- Price order check ---
    order_fail_idx = []
    for i in range(1, len(rows)):
        prev = rows.iloc[i - 1]
        curr = rows.iloc[i]
        if curr[col] == 1:          # current is a high → must be above prev low
            if curr['high'] <= prev['low']:
                order_fail_idx.append(rows.index[i])
        else:                        # current is a low → must be below prev high
            if curr['low'] >= prev['high']:
                order_fail_idx.append(rows.index[i])
    order_violations = rows.loc[order_fail_idx]

    # --- Print summary ---
    label = col
    total = len(rows)
    print(f"=== {label} validation ({total} marks across full dataset) ===")

    if len(bos_violations):
        print(f"  BOS violations : {len(bos_violations)}")
        print(bos_violations[['timestamp', 'high', 'low', col, 'price_at_mark']].to_string())
    else:
        print("  BOS violations : 0  — alternation OK")

    if len(order_violations):
        print(f"  Order violations: {len(order_violations)}")
        print(order_violations[['timestamp', 'high', 'low', col, 'price_at_mark']].to_string())
    else:
        print("  Order violations: 0  — price ordering OK")

    is_valid = len(bos_violations) == 0 and len(order_violations) == 0
    print(f"  Valid: {is_valid}\n")

    return {
        'bos_violations':   bos_violations,
        'order_violations': order_violations,
        'is_valid':         is_valid,
    }


In [11]:
# --- run validation on STR columns ---
str_15m_report = validate_markers(df_15, 'isSTR')
# --- run validation on ITR columns ---
str_45m_report = validate_markers(df_45, 'isSTR')

=== isSTR validation (13720 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True

=== isSTR validation (4774 marks across full dataset) ===
  BOS violations : 0  — alternation OK
  Order violations: 0  — price ordering OK
  Valid: True



In [12]:
STR_WINDOW        = 4
FIFTEEN_MIN_MS    = 900_000
FORTY_FIVE_MIN_MS = 2_700_000

# Shift forward by window → fully causal: mark at bar t only uses data available at t
df_15['str_confirmed'] = df_15['isSTR'].shift(STR_WINDOW).fillna(0).astype('int8')
df_45['str_confirmed'] = df_45['isSTR'].shift(STR_WINDOW).fillna(0).astype('int8')

# Add bucket keys to 5m bars
df_5['ts_15m'] = (df_5['timestamp'] // FIFTEEN_MIN_MS)    * FIFTEEN_MIN_MS
df_5['ts_45m'] = (df_5['timestamp'] // FORTY_FIVE_MIN_MS) * FORTY_FIVE_MIN_MS

# Build lookup: bucket_key → confirmed value
# df_15 timestamps are 15m boundaries → use directly as bucket key
lookup_15m = {int(k) + FIFTEEN_MIN_MS:    v for k, v in df_15.set_index('timestamp')['str_confirmed'].to_dict().items()}

# df_45 timestamps = first 15m bar in bucket; re-derive the 45m bucket key
df_45['ts_45m'] = (df_45['timestamp'] // FORTY_FIVE_MIN_MS) * FORTY_FIVE_MIN_MS
lookup_45m = {int(k) + FORTY_FIVE_MIN_MS: v for k, v in df_45.set_index('ts_45m')['str_confirmed'].to_dict().items()}

# Map to 5m — broad pass (all bars in bucket get the value)
df_5['15STR_confirmed'] = df_5['ts_15m'].map(lookup_15m).fillna(0).astype('int8')
df_5['45STR_confirmed'] = df_5['ts_45m'].map(lookup_45m).fillna(0).astype('int8')

# Keep only the FIRST 5m bar in each bucket (confirmation fires once, at bucket open)
is_first_in_15m = df_5.groupby('ts_15m').cumcount() == 0
is_first_in_45m = df_5.groupby('ts_45m').cumcount() == 0

df_5['15STR_confirmed'] = df_5['15STR_confirmed'].where(is_first_in_15m, 0)
df_5['45STR_confirmed'] = df_5['45STR_confirmed'].where(is_first_in_45m, 0)

print(f"15STR_confirmed  highs: {(df_5['15STR_confirmed']==1).sum():,}  lows: {(df_5['15STR_confirmed']==-1).sum():,}")
print(f"45STR_confirmed  highs: {(df_5['45STR_confirmed']==1).sum():,}  lows: {(df_5['45STR_confirmed']==-1).sum():,}")


15STR_confirmed  highs: 6,797  lows: 6,796
45STR_confirmed  highs: 2,369  lows: 2,369


In [13]:
# preview
del df_45, df_15
print(df_5.head())

       timestamp          open          high           low         close  \
0  1704037500000  42497.851562  42518.148438  42480.691406  42482.250000   
1  1704037800000  42482.238281  42500.000000  42411.101562  42414.421875   
2  1704038100000  42414.429688  42471.988281  42414.421875  42457.171875   
3  1704038400000  42457.171875  42484.828125  42436.468750  42449.601562   
4  1704038700000  42449.589844  42488.000000  42449.589844  42487.988281   

         vol  atr42          ts_5m  label  train_mask  body_pct  vol_norm  \
0  56.512959    NaN  1704037500000      0           0 -0.000367       NaN   
1  47.354488    NaN  1704037800000      0           0 -0.001596       NaN   
2  43.228668    NaN  1704038100000      0           0  0.001008       NaN   
3  74.418266    NaN  1704038400000      0           0 -0.000178       NaN   
4  38.818489    NaN  1704038700000      0           0  0.000905       NaN   

   close_ret1  atr42_pct         ts_15m         ts_45m  15STR_confirmed  \
0    

In [14]:
df_5.head()

,timestamp,open,high,low,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,ts_15m,ts_45m,15STR_confirmed,45STR_confirmed
0,1704037500000,42497.851562,42518.148438,42480.691406,42482.250000,56.512959,NaN,1704037500000,0,0,-0.000367,NaN,NaN,NaN,1704037500000,1704037500000,0,0
1,1704037800000,42482.238281,42500.000000,42411.101562,42414.421875,47.354488,NaN,1704037800000,0,0,-0.001596,NaN,-0.001597,NaN,1704037500000,1704037500000,0,0
2,1704038100000,42414.429688,42471.988281,42414.421875,42457.171875,43.228668,NaN,1704038100000,0,0,0.001008,NaN,0.001008,NaN,1704037500000,1704037500000,0,0
3,1704038400000,42457.171875,42484.828125,42436.468750,42449.601562,74.418266,NaN,1704038400000,0,0,-0.000178,NaN,-0.000178,NaN,1704038400000,1704037500000,0,0
4,1704038700000,42449.589844,42488.000000,42449.589844,42487.988281,38.818489,NaN,1704038700000,0,0,0.000905,NaN,0.000904,NaN,1704038400000,1704037500000,0,0


# EMA Trend
```
close_pct_vs_ema50  = (close - EMA(50))  / close   # ~12.5h context
close_pct_vs_ema200 = (close - EMA(200)) / close 

In [15]:
# def ema(series, period):
#     """Exponential Moving Average — matches TradingView/most platform behavior."""
#     k = 2 / (period + 1)
#     result = np.empty(len(series), dtype="float32")
#     result[:] = np.nan

#     # seed: first non-NaN value
#     start = np.argmax(~np.isnan(series))
#     result[start] = series[start]

#     for i in range(start + 1, len(series)):
#         result[i] = series[i] * k + result[i - 1] * (1 - k)

#     return result
# ltf_arr['c_EMA50'] = ema(ltf_arr['close'],50)
# ltf_arr['c_EMA200'] = ema(ltf_arr['close'],200)
# ltf_arr['c_EMA50_pct'] = (ltf_arr['close']-ltf_arr['c_EMA50'])/ltf_arr['close']
# ltf_arr['c_EMA200_pct'] = (ltf_arr['close']-ltf_arr['c_EMA200'])/ltf_arr['close']

# ltf_arr.drop(columns=["c_EMA200","c_EMA50"], inplace=True)
# print(ltf_arr.columns)
# ltf_arr.head()


# Weekly Range Position (extends existing pattern)
```
week_hi  = rolling_max(high,  672)   # 672 bars = 7 days × 96 bars/day
week_lo  = rolling_min(low,   672)
pct_in_weekly_range = (close - week_lo) / (week_hi - week_lo)  # 0–1
```

In [16]:
# # Weekly hi-lo
# ltf_arr['week_hi'] = ltf_arr['high'].rolling(window=672).max()
# ltf_arr['week_lo'] = ltf_arr['low'].rolling(window=672).min()
# ltf_arr['close_pct_w_range'] = (ltf_arr['close'] - ltf_arr['week_lo']) / (ltf_arr['week_hi'] - ltf_arr['week_lo'])

# ltf_arr.drop(columns=['week_hi','week_lo'], inplace=True)
# print(ltf_arr.columns)
# ltf_arr.head()

# Rolling VWAP Deviation
```
typical_price  = (high + low + close) / 3
vwap_96        = (typical_price * vol).rolling(96).sum() / vol.rolling(96).sum()
close_vs_vwap  = (close - vwap_96) / close
```

In [17]:
# # rolling VWAP
# ltf_arr['hlc3'] = (ltf_arr['high'] + ltf_arr['low'] + ltf_arr['close']) / 3
# ltf_arr['vwap_96'] = (ltf_arr['hlc3'] * ltf_arr['vol']).rolling(window=96).sum() / ltf_arr['vol'].rolling(window=96).sum()
# ltf_arr['c_vwap_96'] = (ltf_arr['close'] - ltf_arr['vwap_96']) / ltf_arr['close']

# ltf_arr.drop(columns=['vwap_96','hlc3'], inplace=True)
# print(ltf_arr.columns)
# ltf_arr.head()

In [18]:
# preview
print(df_5.columns)
df_5.head()

Index(['timestamp', 'open', 'high', 'low', 'close', 'vol', 'atr42', 'ts_5m',
       'label', 'train_mask', 'body_pct', 'vol_norm', 'close_ret1',
       'atr42_pct', 'ts_15m', 'ts_45m', '15STR_confirmed', '45STR_confirmed'],
      dtype='object')


,timestamp,open,high,low,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,ts_15m,ts_45m,15STR_confirmed,45STR_confirmed
0,1704037500000,42497.851562,42518.148438,42480.691406,42482.250000,56.512959,NaN,1704037500000,0,0,-0.000367,NaN,NaN,NaN,1704037500000,1704037500000,0,0
1,1704037800000,42482.238281,42500.000000,42411.101562,42414.421875,47.354488,NaN,1704037800000,0,0,-0.001596,NaN,-0.001597,NaN,1704037500000,1704037500000,0,0
2,1704038100000,42414.429688,42471.988281,42414.421875,42457.171875,43.228668,NaN,1704038100000,0,0,0.001008,NaN,0.001008,NaN,1704037500000,1704037500000,0,0
3,1704038400000,42457.171875,42484.828125,42436.468750,42449.601562,74.418266,NaN,1704038400000,0,0,-0.000178,NaN,-0.000178,NaN,1704038400000,1704037500000,0,0
4,1704038700000,42449.589844,42488.000000,42449.589844,42487.988281,38.818489,NaN,1704038700000,0,0,0.000905,NaN,0.000904,NaN,1704038400000,1704037500000,0,0


# Export Data

In [19]:
# Export marker to json visulize the data on Chart
# save export to jsonl
# Export marker to json visulize the data on Chart
# save export to jsonl

folder_path = Path.cwd().parents[1] / "data" / "mlData"
folder_path.mkdir(parents=True, exist_ok=True)

out_path = folder_path / "BTCUSD-5m-v6.jsonl"
df_5.to_json(out_path, orient="records", lines=True)
print(f"Saved {len(df_5):,} rows → {out_path}")

Saved 228,772 rows → /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-CandleScience/data/mlData/BTCUSD-5m-v6.jsonl
